In [ ]:
%reload_ext autoreload
%autoreload 2
%cd ~/erc-src/cuneiform-ocr-sign-alignment-worktree # Change to the project directory
%env PATH=$HOME/.local/bin:$PATH
    
import json
import os
import requests
import numpy as np
import cv2
from PIL import Image
from pymongo import MongoClient
# from mmdet.apis import init_detector, inference_detector
# from mmdet.utils import register_all_modules
from data_processing.divide_photos import divide_tablet_photo
from sign_alignment.detector import ModelConfig, TabletImageDetector

# import signs_alignment as sa
import os
import torch
from dotenv import load_dotenv

ANNOTATIONS_DIR = os.path.expanduser("~/erc-work-data/data-of-cuneiform-ocr-data/filtered_annotations")
CONFIG_FILE = "configs/detr.py"
CHECKPOINT_FILE = os.path.expanduser("~/erc-work-data/retrained_models/detr-173/epoch_1000.pth")
SCORE_THRESHOLD = 0.5
Y_THRESHOLD = 35  # for grouping signs into lines
OUTPUT_DIR = "alignment_results"
SAMPLE_LIMIT = 5  # number of samples to process

load_dotenv()
MONGODB_URI = os.getenv('MONGODB_URI', 'YOUR_MONGODB_URI')

In [ ]:
model_config = ModelConfig(
    config_file=CONFIG_FILE,
    checkpoint_file=CHECKPOINT_FILE,
    device='auto'
)
tablet_detector = TabletImageDetector(
    model_config=model_config,
    score_threshold=SCORE_THRESHOLD,
    keep_crops=True
)

In [ ]:
from sign_alignment.data_source import LocalDataSource

local_source = LocalDataSource(ANNOTATIONS_DIR)
fragments = local_source.get_available_fragments()
print(f"Found {len(fragments)} fragments with both image and annotation")
sample = fragments[0] # NBC.4020
sample = fragments[9] # HS.2086
print(f"Processing sample: {sample}")


In [ ]:
# show groud truth boxes
from sign_alignment.visualizer import BboxVisualizer

img = local_source.load_image(sample)
gt_boxes = local_source.load_annotation(sample)
print(f"Ground truth boxes: {gt_boxes}")

gt_bbox_visualizer = BboxVisualizer(color=(0, 255, 0)) # green for gt
gt_bbox_visualizer.draw_boxes(img.copy(), gt_boxes)
# gt_bbox_visualizer.display_result(vis_opt = "save", path = )
gt_bbox_visualizer.show_draw()
gt_bbox_visualizer.save(os.path.join(OUTPUT_DIR, f"debug_{sample}_gt.jpg"))


In [ ]:
# sign text
from sign_alignment.data_source import EBLAPISource, SignTextParser
from sign_alignment.visualizer import TextVisualizer

api_source = EBLAPISource()
signs_text = api_source.get_signs(sample)
if signs_text is None:
    raise ValueError(f"No sign text found for sample {sample}")
print(f"  Raw API signs (first 50 chars): {signs_text[:50]}...")

text_lines = SignTextParser.parse_api_signs(signs_text)
total_text_signs = sum(len(line) for line in text_lines)
print(f"  Text lines: {len(text_lines)}, total signs: {total_text_signs}")
# print all text lines
TextVisualizer.save_text(text_lines, path=os.path.join(OUTPUT_DIR, f"debug_{sample}_text.txt"), fragment_id=sample)

In [ ]:
# detect signs
from sign_alignment.visualizer import BboxVisualizer

# Use tablet_detector created in cell 2
detections = tablet_detector.detect(img)

det_bbox_visualizer = BboxVisualizer(color=(255, 0, 0)) # red for detections
det_bbox_visualizer.draw_boxes(img.copy(), detections)
det_bbox_visualizer.display_result(vis_opt = "save", path = os.path.join(OUTPUT_DIR, f"debug_{sample}_det.jpg"))

cropped = tablet_detector.get_cropped_images()
exp_image = cropped[1]


det_bbox_visualizer.draw_boxes(exp_image.img.copy(), exp_image.detections)
det_bbox_visualizer.display_result(vis_opt = "draw", path = os.path.join(OUTPUT_DIR, f"debug_{sample}_exp_image.jpg"))

In [ ]:
# Transform GT boxes for exp_image (cropped region)
from sign_alignment import transform_gt_to_cropped_region

# exp_image is cropped[3], so we need crop_coordinates[3]
exp_image_idx = 3
crop_info = tablet_detector.crop_coordinates[exp_image_idx]

print(f"Crop info for exp_image (index {exp_image_idx}): x={crop_info['x']}, y={crop_info['y']}, w={crop_info['w']}, h={crop_info['h']}")

# Transform GT boxes to exp_image coordinates
gt_boxes_exp = transform_gt_to_cropped_region(gt_boxes, crop_info)
print(f"GT boxes in full image: {len(gt_boxes)}")
print(f"GT boxes in exp_image: {len(gt_boxes_exp)}")

# Visualize GT boxes on exp_image
if gt_boxes_exp:
    gt_exp_bbox_visualizer = BboxVisualizer(color=(0, 255, 255))  # cyan for gt
    gt_exp_bbox_visualizer.draw_boxes(exp_image.img.copy(), gt_boxes_exp)
    gt_exp_bbox_visualizer.display_result(vis_opt="save", path=os.path.join(OUTPUT_DIR, f"debug_{sample}_exp_gt.jpg"))

In [ ]:
# statistics
from sign_alignment import compute_avg_dimensions

avg_width, avg_height = compute_avg_dimensions(detections)
print(f"full image shape: {img.shape}")
print(f"Exp image shape: {exp_image.img.shape}")
print(f"Average detected sign width: {avg_width:.2f}, height: {avg_height:.2f}")

In [ ]:
# Build heatmap for position probability of each class
from sign_alignment import create_detection_heatmap, CLASSES_ABZ

scale_factor = 10

# Create detection heatmap
heatmap, influence_radius, sigma = create_detection_heatmap(
    exp_image.detections, 
    exp_image.img.shape, 
    scale_factor=scale_factor,
    avg_width=avg_width,
    avg_height=avg_height,
    method='gaussian'
)

print(f"Original image shape: {exp_image.img.shape[:2]}")
print(f"Created heatmap with shape: {heatmap.shape}")
print(f"Scale factor: 1/{scale_factor}")
print(f"Number of CLASSES_ABZ: {len(CLASSES_ABZ)}")
print(f"Influence radius (scaled): {influence_radius:.2f}, sigma: {sigma:.2f}")
print(f"Heatmap generated with {len(exp_image.detections)} detections")
print(f"Heatmap value range: [{heatmap.min():.3f}, {heatmap.max():.3f}]")

In [ ]:
# Visualize exp image heatmap 
from sign_alignment import HeatmapVisualizer, SignResolver

heatmap_visualizer = HeatmapVisualizer()

# find class id for sign name "A"
class_id_for_A = None
class_id_for_NA = None
for i, abz_name in enumerate(CLASSES_ABZ):
    if SignResolver.from_abz(abz_name).name == "A":
        class_id_for_A = i
        break
for i, abz_name in enumerate(CLASSES_ABZ):
    if SignResolver.from_abz(abz_name).name == "NA":
        class_id_for_NA = i
        break
print(f"Class ID for sign 'A': {class_id_for_A}")
print(f"Class ID for sign 'NA': {class_id_for_NA}")
    
heatmap_visualizer.draw_channels(exp_image.img, heatmap, channels=(class_id_for_A, class_id_for_NA, 2), detection=exp_image.detections)
heatmap_visualizer.display_result(vis_opt="draw")
heatmap_visualizer.draw_pca(exp_image.img, heatmap, detection=exp_image.detections)
heatmap_visualizer.display_result(vis_opt="draw")

In [ ]:
# Create heatmap from text_lines with synthetic sign positions
from sign_alignment import create_text_heatmap

# Create text heatmap (classes_abz defaults to CLASSES_ABZ)
heatmap_text, margin, influence_radius_text, sigma_text = create_text_heatmap(
    text_lines, 
    avg_width, 
    avg_height, 
    scale_factor=scale_factor,
    method='gaussian'
)

max_row_length = max(len(line) for line in text_lines)
num_rows = len(text_lines)
total_signs_in_text = sum(len(line) for line in text_lines)

print(f"Text lines grid: {num_rows} rows, max {max_row_length} columns")
print(f"Average sign dimensions: width={avg_width:.2f}, height={avg_height:.2f}")
print(f"Text heatmap dimensions (after scaling): {heatmap_text.shape}")
print(f"Margin used: {margin:.2f}")
print(f"Gaussian parameters - influence radius: {influence_radius_text:.2f}, sigma: {sigma_text:.2f}")
print(f"Heatmap generated for {total_signs_in_text} signs from text_lines")
print(f"Heatmap value range: [{heatmap_text.min():.3f}, {heatmap_text.max():.3f}]")

In [ ]:
# Visualize text_lines heatmap

heatmap_visualizer_text = HeatmapVisualizer()
heatmap_visualizer_text.draw_channels(None, heatmap_text, 
                                      channels=(class_id_for_A, class_id_for_NA, 2), 
                                      detection=None, text_lines=text_lines)
heatmap_visualizer_text.display_result(vis_opt="draw")

In [ ]:
# 1. Use normalized cross-correlation to find template position
from sign_alignment import match_heatmaps_ncc

print(f"Exp heatmap shape: {heatmap.shape}")
print(f"Text heatmap shape: {heatmap_text.shape}")

top_left_scaled, max_val, top_left_original = match_heatmaps_ncc(
    heatmap, 
    heatmap_text, 
    scale_factor=scale_factor
)

top_left_x_text = top_left_original[0]
top_left_y_text = top_left_original[1]

print(f"Best match correlation (averaged over {heatmap.shape[2]} classes): {max_val:.4f}")
print(f"Match position in text heatmap (scaled): {top_left_scaled}")
print(f"Match position in text heatmap (original): ({top_left_x_text:.2f}, {top_left_y_text:.2f})")

# Convert to full tablet image coordinates
top_left_x_tablet = top_left_x_text - margin
top_left_y_tablet = top_left_y_text - margin

print(f"Match position in full tablet image: ({top_left_x_tablet:.2f}, {top_left_y_tablet:.2f})")

In [ ]:
# 2. Visualize the position of exp image heatmap in text heatmap

# Create visualization using the mixed heatmaps for display
heatmap_exp_mixed = np.mean(heatmap, axis=2)
heatmap_text_mixed = np.mean(heatmap_text, axis=2)

vis_text_heatmap = cv2.normalize(heatmap_text_mixed, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
vis_text_heatmap_rgb = cv2.cvtColor(vis_text_heatmap, cv2.COLOR_GRAY2RGB)

# Draw rectangle at matched position (in scaled coordinates)
bottom_right_scaled = (top_left_scaled[0] + heatmap_exp_mixed.shape[1], 
                       top_left_scaled[1] + heatmap_exp_mixed.shape[0])
cv2.rectangle(vis_text_heatmap_rgb, top_left_scaled, bottom_right_scaled, (0, 255, 0), 2)

# Display
import matplotlib.pyplot as plt
plt.figure(figsize=(15, 10))
plt.imshow(vis_text_heatmap_rgb)
plt.title(f"Text Heatmap with Matched Exp Image Position (green box)\nCorrelation: {max_val:.4f}")
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"Green box shows matched position of exp image in text heatmap")

In [ ]:
# 3. Assign sign centers from text image into exp image and create bboxes
from sign_alignment import create_text_based_detections

exp_img_height, exp_img_width = exp_image.img.shape[:2]

detection_with_texts = create_text_based_detections(
    text_lines, 
    top_left_x_text, 
    top_left_y_text, 
    margin, 
    avg_width, 
    avg_height, 
    (exp_img_width, exp_img_height)
)

print(f"Created {len(detection_with_texts)} text-based detections for exp image")
print(f"Original detections: {len(exp_image.detections)}")

# Show first few detections as example
if detection_with_texts:
    print("\nFirst 3 text-based detections:")
    for i, det in enumerate(detection_with_texts[:3]):
        print(f"  {i+1}. {det.sign.abz}: bbox=[{det.x1:.1f}, {det.y1:.1f}, {det.x2:.1f}, {det.y2:.1f}], "
              f"center=({(det.x1+det.x2)/2:.1f}, {(det.y1+det.y2)/2:.1f})")

In [ ]:
# 4. Visualize detection_with_texts for exp image

text_det_bbox_visualizer = BboxVisualizer(color=(0, 0, 255))  # blue for text-based detections
text_det_bbox_visualizer.draw_boxes(exp_image.img.copy(), detection_with_texts)
text_det_bbox_visualizer.display_result(vis_opt="draw", path=os.path.join(OUTPUT_DIR, f"debug_{sample}_text_detections.jpg"))

print(f"Visualized {len(detection_with_texts)} text-based detections on exp image")

In [ ]:
# 5. Organize data using SubTablet data structures
from sign_alignment import SubTablet

# --- 1. Create SubTablet for detection results (exp_image) ---
sub_tablet_detection = SubTablet.from_detections(
    img=exp_image.img,
    detections=exp_image.detections,
    name="detection",
    avg_width=avg_width,
    avg_height=avg_height
)

# Generate heatmap for detection
sub_tablet_detection.create_heatmap(scale_factor=scale_factor, method='gaussian')

print(f"=== SubTablet: Detection ===")
print(f"  Name: {sub_tablet_detection.name}")
print(f"  Image shape: {sub_tablet_detection.shape}")
print(f"  Num sign boxes: {len(sub_tablet_detection)}")
print(f"  Avg dimensions: {sub_tablet_detection.avg_width:.2f} x {sub_tablet_detection.avg_height:.2f}")
print(f"  Heatmap shape: {sub_tablet_detection.heatmap.shape}")

# --- 2. Create SubTablet for full text (hypothetical full tablet from API text) ---
sub_tablet_full_text = SubTablet.from_text_lines(
    text_lines=text_lines,
    avg_width=avg_width,
    avg_height=avg_height,
    margin=margin,
    name="full_text"
)

# Generate heatmap for full text
sub_tablet_full_text.create_heatmap(scale_factor=scale_factor, method='gaussian')

print(f"\n=== SubTablet: Full Text ===")
print(f"  Name: {sub_tablet_full_text.name}")
print(f"  Estimated shape: {sub_tablet_full_text.shape}")
print(f"  Num sign boxes: {len(sub_tablet_full_text)}")
print(f"  Margin: {sub_tablet_full_text.margin:.2f}")
print(f"  Heatmap shape: {sub_tablet_full_text.heatmap.shape}")

# --- 3. Create SubTablet for text-aligned signs ---
sub_tablet_text_aligned = sub_tablet_full_text.extract_sub_region(
    offset_x=top_left_x_text,
    offset_y=top_left_y_text,
    width=exp_img_width,
    height=exp_img_height,
    img=exp_image.img,
    name="text_aligned"
)

# Generate heatmap for text-aligned (using image dimensions)
sub_tablet_text_aligned.create_heatmap(
    scale_factor=scale_factor,
    img_shape=exp_image.img.shape,
    method='gaussian'
)

print(f"\n=== SubTablet: Text Aligned ===")
print(f"  Name: {sub_tablet_text_aligned.name}")
print(f"  Image shape: {sub_tablet_text_aligned.shape}")
print(f"  Num sign boxes: {len(sub_tablet_text_aligned)}")
print(f"  Origin offset: ({sub_tablet_text_aligned.origin_x:.2f}, {sub_tablet_text_aligned.origin_y:.2f})")
print(f"  Heatmap shape: {sub_tablet_text_aligned.heatmap.shape}")

# --- Verify consistency ---
print(f"\n=== Verification ===")
print(f"  detection_with_texts count: {len(detection_with_texts)}")
print(f"  sub_tablet_text_aligned count: {len(sub_tablet_text_aligned)}")

print(f"\n  First 3 from sub_tablet_detection:")
for i, sb in enumerate(sub_tablet_detection.sign_boxes[:3]):
    print(f"    {i+1}. {sb.sign_name}: center=({sb.cx:.1f}, {sb.cy:.1f}), size=({sb.width:.1f}, {sb.height:.1f})")

print(f"\n  First 3 from sub_tablet_text_aligned:")
for i, sb in enumerate(sub_tablet_text_aligned.sign_boxes[:3]):
    print(f"    {i+1}. {sb.sign_name}: center=({sb.cx:.1f}, {sb.cy:.1f}), size=({sb.width:.1f}, {sb.height:.1f})")

In [ ]:
# 6. Visualize SubTablet data using existing visualizers
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Visualize detection SubTablet
det_vis = BboxVisualizer(color=(255, 0, 0))  # Red for detection
det_vis.draw_boxes(sub_tablet_detection.img.copy(), sub_tablet_detection.to_detection_list())
axes[0].imshow(cv2.cvtColor(det_vis.result, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Detection ({len(sub_tablet_detection)} signs)")
axes[0].axis('off')

# Visualize text-aligned SubTablet
text_vis = BboxVisualizer(color=(0, 0, 255))  # Blue for text-aligned
text_vis.draw_boxes(sub_tablet_text_aligned.img.copy(), sub_tablet_text_aligned.to_detection_list())
axes[1].imshow(cv2.cvtColor(text_vis.result, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Text Aligned ({len(sub_tablet_text_aligned)} signs)")
axes[1].axis('off')

# Overlay both on same image
det_overlay = BboxVisualizer(color=(255, 0, 0))  # Red for detection
det_overlay.draw_boxes(sub_tablet_detection.img.copy(), sub_tablet_detection.to_detection_list())
text_overlay = BboxVisualizer(color=(0, 0, 255))  # Blue for text
text_overlay.draw_boxes(det_overlay.result, sub_tablet_text_aligned.to_detection_list())
axes[2].imshow(cv2.cvtColor(text_overlay.result, cv2.COLOR_BGR2RGB))
axes[2].set_title(f"Overlay: Detection (red) + Text Aligned (blue)")
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("Red = Detection results, Blue = Text-aligned bboxes")

heatmap_visualizer_opt = HeatmapVisualizer(bbox_color=(200, 0, 200))
heatmap_visualizer_opt.draw_pca(sub_tablet_detection.img, sub_tablet_detection.heatmap, detection=sub_tablet_detection.to_detection_list())
heatmap_visualizer_opt.display_result(vis_opt="draw")
heatmap_visualizer_opt.draw_channels(sub_tablet_detection.img, sub_tablet_detection.heatmap, channels=(class_id_for_A, class_id_for_NA, 2), detection=sub_tablet_detection.to_detection_list())
heatmap_visualizer_opt.display_result(vis_opt="draw")

heatmap_visualizer_opt.draw_pca(sub_tablet_text_aligned.img, sub_tablet_text_aligned.heatmap, detection=sub_tablet_text_aligned.to_detection_list())
heatmap_visualizer_opt.display_result(vis_opt="draw")
heatmap_visualizer_opt.draw_channels(sub_tablet_text_aligned.img, sub_tablet_text_aligned.heatmap, channels=(class_id_for_A, class_id_for_NA, 2), detection=sub_tablet_text_aligned.to_detection_list())
heatmap_visualizer_opt.display_result(vis_opt="draw")

In [ ]:
# 7. Elastic Chain Optimization - Refine text-aligned bboxes using detection heatmap
from sign_alignment import ElasticChainOptimizer, build_agnostic_heatmap

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Build and visualize the class-agnostic existence map
heatmap_agnostic = build_agnostic_heatmap(sub_tablet_detection.heatmap)
print(f"Class-agnostic heatmap shape: {heatmap_agnostic.shape}")
print(f"  value range: [{heatmap_agnostic.min():.4f}, {heatmap_agnostic.max():.4f}]")

optimizer = ElasticChainOptimizer(
    sub_tablet_text=sub_tablet_text_aligned,
    detection_heatmap=sub_tablet_detection.heatmap,
    detection_boxes=sub_tablet_detection.to_detection_list(),
    scale_factor=scale_factor,
    lambda_data=50000.0,
    lambda_iou=20000.0,
    lambda_seq=0.10,
    lambda_smooth=0.05,
    lambda_anchor=0.1,
    alpha_geo=0.0,            # 20% geometric (existence), 80% semantic (per-class)
    prior_aspect_ratio=avg_width / avg_height,
    device=device
)

print(f"=== Elastic Chain Optimizer Initialized ===")
print(f"  Device: {optimizer.device}")
print(f"  Number of signs: {optimizer.num_signs}")
print(f"  Number of rows: {optimizer.num_rows}")
print(f"  Prior aspect ratio: {optimizer.prior_aspect_ratio:.3f}")
print(f"  Heatmap shape: {optimizer.heatmap.shape}")
print(f"  alpha_geo: {optimizer.alpha_geo} (data = {1-optimizer.alpha_geo:.2f}*semantic + {optimizer.alpha_geo:.2f}*geometric)")
print(f"  Agnostic heatmap shape: {optimizer.heatmap_agnostic.shape}")
print(f"  lambda_iou: {optimizer.lambda_iou}")
print(f"  Detection box classes for IoU: {len(optimizer.det_boxes_by_class)}")

# Run optimization
sub_tablet_optimized = optimizer.optimize(
    num_iterations=100,
    lr=5.0,
    verbose=True,
    log_every=5
)

In [ ]:
# 7.5 Debug: Analyze data loss gradient and heatmap values (semantic + geometric + IoU)

test_optimizer = ElasticChainOptimizer(
    sub_tablet_text=sub_tablet_text_aligned,
    detection_heatmap=sub_tablet_detection.heatmap,
    detection_boxes=sub_tablet_detection.to_detection_list(),
    scale_factor=scale_factor,
    lambda_data=1.0,
    lambda_iou=1.0,
    lambda_seq=0.0,
    lambda_smooth=0.0,
    lambda_anchor=0.0,
    alpha_geo=0.0,
    device=device
)

# Compute combined data loss and gradients
loss = test_optimizer.compute_data_loss()
loss.backward()

grads = test_optimizer.params.grad
print("=== Data Loss Gradient Analysis (combined semantic + geometric) ===")
print(f"L_data = {loss.item():.4f}")
print(f"Gradient statistics:")
print(f"  cx grad: mean={grads[:, 0].mean().item():.6f}, std={grads[:, 0].std().item():.6f}, max={grads[:, 0].abs().max().item():.6f}")
print(f"  cy grad: mean={grads[:, 1].mean().item():.6f}, std={grads[:, 1].std().item():.6f}, max={grads[:, 1].abs().max().item():.6f}")
print(f"  w grad:  mean={grads[:, 2].mean().item():.6f}, std={grads[:, 2].std().item():.6f}")
print(f"  h grad:  mean={grads[:, 3].mean().item():.6f}, std={grads[:, 3].std().item():.6f}")

# Compute sub-losses separately for analysis
with torch.no_grad():
    L_sem = test_optimizer.compute_semantic_loss()
    L_geo = test_optimizer.compute_geometric_loss()
    L_iou = test_optimizer.compute_iou_loss()
    print(f"\nL_semantic (per-class) = {L_sem.item():.4f}")
    print(f"L_geometric (agnostic) = {L_geo.item():.4f}")
    print(f"L_iou (shape regression) = {L_iou.item():.4f}")
    print(f"alpha_geo = {test_optimizer.alpha_geo}")
    print(f"Weighted: {(1-test_optimizer.alpha_geo):.2f}*{L_sem.item():.4f} + {test_optimizer.alpha_geo:.2f}*{L_geo.item():.4f} = {loss.item():.4f}")
    print(f"IoU classes matched: {len(test_optimizer.det_boxes_by_class)}")

# Analyze IoU per class
print("\n=== Per-Class IoU Analysis (text-aligned vs detection) ===")
with torch.no_grad():
    params = test_optimizer.params
    for cid, det_tensor in sorted(test_optimizer.det_boxes_by_class.items()):
        # Find optimized signs of this class
        opt_mask = [i for i, c in enumerate(test_optimizer.class_ids) if c == cid]
        if not opt_mask:
            continue
        idx = torch.tensor(opt_mask, device=test_optimizer.device)
        cx, cy, w, h = params[idx, 0], params[idx, 1], params[idx, 2], params[idx, 3]
        opt_x1 = (cx - w / 2).min()
        opt_y1 = (cy - h / 2).min()
        opt_x2 = (cx + w / 2).max()
        opt_y2 = (cy + h / 2).max()
        det_x1 = det_tensor[:, 0].min()
        det_y1 = det_tensor[:, 1].min()
        det_x2 = det_tensor[:, 2].max()
        det_y2 = det_tensor[:, 3].max()
        inter_x1 = torch.max(opt_x1, det_x1)
        inter_y1 = torch.max(opt_y1, det_y1)
        inter_x2 = torch.min(opt_x2, det_x2)
        inter_y2 = torch.min(opt_y2, det_y2)
        inter = torch.clamp(inter_x2 - inter_x1, min=0) * torch.clamp(inter_y2 - inter_y1, min=0)
        area_opt = (opt_x2 - opt_x1) * (opt_y2 - opt_y1)
        area_det = (det_x2 - det_x1) * (det_y2 - det_y1)
        union = area_opt + area_det - inter
        iou = (inter / (union + 1e-6)).item()
        class_name = test_optimizer.classes_abz[cid] if cid < len(test_optimizer.classes_abz) else f"cls_{cid}"
        print(f"  {class_name}: IoU={iou:.4f}, #opt={len(opt_mask)}, #det={len(det_tensor)}")

# Analyze heatmap values at sign positions: per-class + agnostic
print("\n=== Heatmap Values at Text-Aligned Sign Positions ===")
heatmap_h, heatmap_w = sub_tablet_detection.heatmap.shape[:2]
values_semantic = []
values_agnostic = []

for i, sb in enumerate(sub_tablet_text_aligned.sign_boxes[:10]):
    if sb.abz_name in CLASSES_ABZ:
        class_id = CLASSES_ABZ.index(sb.abz_name)
    else:
        continue
    
    sx = int(sb.cx / scale_factor)
    sy = int(sb.cy / scale_factor)
    
    if 0 <= sy < heatmap_h and 0 <= sx < heatmap_w:
        val_sem = sub_tablet_detection.heatmap[sy, sx, class_id]
        val_agn = heatmap_agnostic[sy, sx]
        values_semantic.append(val_sem)
        values_agnostic.append(val_agn)
        print(f"  {i+1}. {sb.sign_name} (class_id={class_id}): pos=({sx},{sy}), "
              f"semantic={val_sem:.4f}, agnostic={val_agn:.4f}")

print(f"\nSemantic - mean: {np.mean(values_semantic):.4f}, max: {np.max(values_semantic):.4f}")
print(f"Agnostic - mean: {np.mean(values_agnostic):.4f}, max: {np.max(values_agnostic):.4f}")

# Check heatmap max values for these classes
print("\n=== Max Heatmap Values for Each Class ===")
for i, sb in enumerate(sub_tablet_text_aligned.sign_boxes[:5]):
    if sb.abz_name in CLASSES_ABZ:
        class_id = CLASSES_ABZ.index(sb.abz_name)
        max_val = sub_tablet_detection.heatmap[:, :, class_id].max()
        print(f"  {sb.sign_name} (class_id={class_id}): max_heatmap={max_val:.4f}")
print(f"Agnostic map max: {heatmap_agnostic.max():.4f}")

In [ ]:
# 7.6 Visualize class-agnostic existence map vs per-class heatmaps
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Class-agnostic existence map with detection boxes
agn_vis = cv2.normalize(heatmap_agnostic, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
agn_color = cv2.applyColorMap(agn_vis, cv2.COLORMAP_JET)
det_vis_agn = BboxVisualizer(color=(255, 255, 255))
det_vis_agn.draw_boxes(
    cv2.resize(agn_color, (exp_image.img.shape[1], exp_image.img.shape[0])),
    sub_tablet_detection.to_detection_list(),
    show_labels=False
)
axes[0, 0].imshow(cv2.cvtColor(det_vis_agn.result, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title(f"H_agnostic (existence map)\nmax={heatmap_agnostic.max():.3f}")
axes[0, 0].axis('off')

# 2. Original + H_agnostic overlay
agn_upscaled = cv2.resize(agn_color, (exp_image.img.shape[1], exp_image.img.shape[0]))
blended = cv2.addWeighted(exp_image.img, 0.5, agn_upscaled, 0.5, 0)
axes[0, 1].imshow(cv2.cvtColor(blended, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title("Original + H_agnostic overlay")
axes[0, 1].axis('off')

# 3. Per-class heatmap PCA false-RGB (computed inline)
H, W, C = sub_tablet_detection.heatmap.shape
hm_flat = sub_tablet_detection.heatmap.reshape(-1, C)
pca = PCA(n_components=3)
pca_result = pca.fit_transform(hm_flat).reshape(H, W, 3)
false_rgb = np.zeros((H, W, 3), dtype=np.float32)
for i in range(3):
    ch = pca_result[:, :, i]
    ch_min, ch_max = ch.min(), ch.max()
    false_rgb[:, :, i] = (ch - ch_min) / (ch_max - ch_min) if ch_max > ch_min else 0
ev = pca.explained_variance_ratio_[:3]
axes[1, 0].imshow(false_rgb)
axes[1, 0].set_title(f"Per-class PCA false-RGB\nR:{ev[0]:.1%} G:{ev[1]:.1%} B:{ev[2]:.1%}")
axes[1, 0].axis('off')

# 4. PCA overlay on original image
img_resized = cv2.resize(cv2.cvtColor(exp_image.img, cv2.COLOR_BGR2RGB),
                          (W, H)).astype(np.float32) / 255.0
pca_blended = np.clip(img_resized * 0.4 + false_rgb * 0.6, 0, 1)
axes[1, 1].imshow(pca_blended)
axes[1, 1].set_title("Original + PCA overlay")
axes[1, 1].axis('off')

plt.suptitle("Class-Agnostic Existence Map: H_agnostic(x,y) = max_c ScoreMap_c(x,y)", fontsize=13)
plt.tight_layout()
plt.show()

print("Top: agnostic existence map captures 'something is here' regardless of class label.")
print("Bottom: per-class PCA shows class-specific spatial distribution.")

In [ ]:
# 8. Visualize Optimization Results

# Plot loss history
optimizer.plot_loss_history()

# Visualize comparison: before vs after optimization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Before: Text-aligned (blue)
before_vis = BboxVisualizer(color=(0, 0, 255))
before_vis.draw_boxes(sub_tablet_text_aligned.img.copy(), sub_tablet_text_aligned.to_detection_list())
axes[0, 0].imshow(cv2.cvtColor(before_vis.result, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title(f"Before: Text Aligned ({len(sub_tablet_text_aligned)} signs)")
axes[0, 0].axis('off')

# After: Optimized (cyan)
after_vis = BboxVisualizer(color=(0, 255, 255))
after_vis.draw_boxes(sub_tablet_optimized.img.copy(), sub_tablet_optimized.to_detection_list())
axes[0, 1].imshow(cv2.cvtColor(after_vis.result, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title(f"After: Optimized ({len(sub_tablet_optimized)} signs)")
axes[0, 1].axis('off')

# Overlay: Detection (red) + Optimized (cyan)
det_vis_overlay = BboxVisualizer(color=(255, 0, 0))
det_vis_overlay.draw_boxes(sub_tablet_detection.img.copy(), sub_tablet_detection.to_detection_list())
opt_vis_overlay = BboxVisualizer(color=(0, 255, 255))
opt_vis_overlay.draw_boxes(det_vis_overlay.result, sub_tablet_optimized.to_detection_list())
axes[1, 0].imshow(cv2.cvtColor(opt_vis_overlay.result, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title(f"Overlay: Detection (red) + Optimized (cyan)")
axes[1, 0].axis('off')

# Overlay: Optimized + Ground Truth
gt_vis_overlay = BboxVisualizer(color=(0, 255, 0))  # green for ground truth
gt_vis_overlay.draw_boxes(sub_tablet_optimized.img.copy(), gt_boxes_exp)
opt_vis_overlay_gt = BboxVisualizer(color=(0, 255, 255))  # cyan for optimized
opt_vis_overlay_gt.draw_boxes(gt_vis_overlay.result, sub_tablet_optimized.to_detection_list())
axes[1, 1].imshow(cv2.cvtColor(opt_vis_overlay_gt.result, cv2.COLOR_BGR2RGB))
axes[1, 1].set_title(f"Overlay: Optimized (cyan) + Ground Truth (green)")
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print("Blue = Text-aligned (before), Cyan = Optimized (after), Red = Detection, Green = Ground Truth")

In [ ]:
# 9. Analyze Parameter Changes

# Get parameter changes
param_changes = optimizer.get_param_changes()

print("=== Parameter Changes Summary ===")
print(f"  Mean delta_cx: {param_changes[:, 0].mean():.2f} (std: {param_changes[:, 0].std():.2f})")
print(f"  Mean delta_cy: {param_changes[:, 1].mean():.2f} (std: {param_changes[:, 1].std():.2f})")
print(f"  Mean delta_w:  {param_changes[:, 2].mean():.2f} (std: {param_changes[:, 2].std():.2f})")
print(f"  Mean delta_h:  {param_changes[:, 3].mean():.2f} (std: {param_changes[:, 3].std():.2f})")

# Compare first few signs before/after
print("\n=== First 5 Signs: Before vs After ===")
for i in range(min(5, len(sub_tablet_text_aligned.sign_boxes))):
    before = sub_tablet_text_aligned.sign_boxes[i]
    after = sub_tablet_optimized.sign_boxes[i]
    print(f"  {i+1}. {before.sign_name}:")
    print(f"      Before: cx={before.cx:.1f}, cy={before.cy:.1f}, w={before.width:.1f}, h={before.height:.1f}")
    print(f"      After:  cx={after.cx:.1f}, cy={after.cy:.1f}, w={after.width:.1f}, h={after.height:.1f}")
    print(f"      Delta:  Δcx={after.cx-before.cx:.1f}, Δcy={after.cy-before.cy:.1f}, Δw={after.width-before.width:.1f}, Δh={after.height-before.height:.1f}")